In [ ]:
import os
import sys

from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
import SimpleITK as sitk
import nrrd
import vtk

import torch
from torch import nn
from torch.nn import functional as F
from torchvision import transforms

import pytorch_lightning as pl
import pickleπ
import monai 
import glob 
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D


sys.path.append('/mnt/raid/C1_ML_Analysis/source/famli-ultra-sim/')
sys.path.append('/mnt/raid/C1_ML_Analysis/source/famli-ultra-sim/dl')
import dl.loaders.ultrasound_dataset as usd
import dl.transforms.ultrasound_transforms as ultrasound_transforms
import dl.nets.us_simu as us_simu

In [2]:
args = {
    'batch_size': 2,
    'num_workers': 6,
    'max_sweeps': 2,
    'max_sweeps_val': -1,
    'num_frames': 96,
    'img_column': "file_path",
    'tag_column': "tag",
    'frame_column': "frame",
    'id_column': "study_id",
    'ga_column': None,
    'csv_train': '/mnt/raid/C1_ML_Analysis/simulated_data_export/animation_export/all_poses_sweeps_us_frame_train_train.csv',
    'csv_valid': '/mnt/raid/C1_ML_Analysis/simulated_data_export/animation_export/all_poses_sweeps_us_frame_train_test.csv',
    'csv_test': '/mnt/raid/C1_ML_Analysis/simulated_data_export/animation_export/all_poses_sweeps_us_frame_test.csv',
    'mount_point': "/mnt/raid/C1_ML_Analysis/simulated_data_export/animation_export",
    'drop_last': 0,
    'class_column': "presentation",
}

dm = usd.USDataModuleBlindSweepWTag(**args)

In [3]:

args = {
        'lr': 1e-4,
        'weight_decay': 0.01,
        'spatial_dims': 2,
        'in_channels': 1,
        'features': 1280,
        'n_chunks_e': 16,
        'n_chunks': 16,
        'num_heads': 8,
        'num_samples': 4096,
        'input_dim': 3,
        'embed_dim': 128,
        'output_dim': 3,
        'dropout': 0.25,
        'time_steps': 96,
        'tags': [0, 1, 2, 3, 4, 5, 6, 7, 8],  
}

model = us_simu.USBabyFrame(**args)

In [4]:
dm.setup()
train_loader = dm.train_dataloader()

In [5]:
batch = next(iter(train_loader))

In [6]:
model.training_step(batch, 0)

/mnt/raid/home/jprieto/envs/torch_us/lib/python3.10/site-packages/lightning/pytorch/core/module.py:441: You are trying to `self.log()` but the `self.trainer` reference is not registered on the model yet. This is most likely because the model hasn't been passed to the `Trainer`


metatensor(0.9023, grad_fn=<AliasBackward0>)

In [8]:
dim_time = 9
mask = torch.ones(dim_time, dim_time)
src = torch.zeros(dim_time, 3)
index_tensor = torch.stack([torch.arange(i-1,i+2) for i in range(dim_time)], dim = 0)
#mask = torch.triu(mask,diagonal=0).T
#mask = mask.fill_diagonal_(0)
index_tensor[index_tensor < 0] = 0
index_tensor[index_tensor >= dim_time] = dim_time - 1
mask.scatter_(1, index_tensor, src)

tensor([[0., 0., 1., 1., 1., 1., 1., 1., 1.],
        [0., 0., 0., 1., 1., 1., 1., 1., 1.],
        [1., 0., 0., 0., 1., 1., 1., 1., 1.],
        [1., 1., 0., 0., 0., 1., 1., 1., 1.],
        [1., 1., 1., 0., 0., 0., 1., 1., 1.],
        [1., 1., 1., 1., 0., 0., 0., 1., 1.],
        [1., 1., 1., 1., 1., 0., 0., 0., 1.],
        [1., 1., 1., 1., 1., 1., 0., 0., 0.],
        [1., 1., 1., 1., 1., 1., 1., 0., 0.]])

In [9]:
torch.triu(torch.full((dim_time, dim_time), float("-inf")), diagonal=1)

tensor([[0., -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf],
        [0., 0., -inf, -inf, -inf, -inf, -inf, -inf, -inf],
        [0., 0., 0., -inf, -inf, -inf, -inf, -inf, -inf],
        [0., 0., 0., 0., -inf, -inf, -inf, -inf, -inf],
        [0., 0., 0., 0., 0., -inf, -inf, -inf, -inf],
        [0., 0., 0., 0., 0., 0., -inf, -inf, -inf],
        [0., 0., 0., 0., 0., 0., 0., -inf, -inf],
        [0., 0., 0., 0., 0., 0., 0., 0., -inf],
        [0., 0., 0., 0., 0., 0., 0., 0., 0.]])